# NLA Verbalizer — Batch
Lee `metadatos_activaciones.json` + archivos `.npy` producidos por `NLA_inferencia_batch.ipynb`,
verbaliza cada activación con el AV y guarda todas las explicaciones en un único CSV.

In [ ]:
import torch, numpy as np, json, os, yaml, re, csv
from google.colab import drive
drive.mount('/content/drive')


# ── Rutas ── deben coincidir con las de NLA_inferencia_batch.ipynb
HACKATHON      = '/content/drive/MyDrive/HACKATHON'
CHECKPOINT_AV  = '/content/drive/MyDrive/nla_pipeline/checkpoints/nla_av'
DIR_ACT        = f'{HACKATHON}/activaciones'
METADATA_JSON  = f'{DIR_ACT}/metadatos_activaciones.json'
CSV_SALIDA     = f'{HACKATHON}/explicaciones_nla.csv'

# Cargar metadatos del batch de inferencia
with open(METADATA_JSON, encoding='utf-8') as f:
    metadatos = json.load(f)

print(f'Entradas cargadas : {len(metadatos)}')
print(f'Grupos            : {set(m["grupo"] for m in metadatos)}')
print(f'Idiomas           : {set(m["lang"]  for m in metadatos)}')
print(f'Carpeta .npy      : {DIR_ACT}')
print(f'CSV de salida     : {CSV_SALIDA}')

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
# Leer configuración del sidecar del AV
with open(f'{CHECKPOINT_AV}/nla_meta.yaml') as f:
    nla_meta = yaml.safe_load(f)

D_MODEL          = nla_meta['d_model']
INJECTION_SCALE  = nla_meta['extraction']['injection_scale']
INJECTION_CHAR   = nla_meta['tokens']['injection_char']
INJECTION_TOK_ID = nla_meta['tokens']['injection_token_id']
LEFT_NEIGHBOR    = nla_meta['tokens']['injection_left_neighbor_id']
RIGHT_NEIGHBOR   = nla_meta['tokens']['injection_right_neighbor_id']
PROMPT_TEMPLATE  = nla_meta['prompt_templates']['av']

print(f'd_model          : {D_MODEL}')
print(f'injection_scale  : {INJECTION_SCALE}')
print(f'injection_char   : {repr(INJECTION_CHAR)}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from safetensors import safe_open

vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20
print(f'VRAM: {vram_gb:.1f} GB → Modo: {"8-bit" if MODO_8BIT else "bfloat16"}')

tok_av = AutoTokenizer.from_pretrained(CHECKPOINT_AV, trust_remote_code=True)

ids_test = tok_av.encode(INJECTION_CHAR, add_special_tokens=False)
assert ids_test == [INJECTION_TOK_ID], \
    f'ERROR: {INJECTION_CHAR} → {ids_test}, esperado [{INJECTION_TOK_ID}]'
print(f'✓ Carácter de inyección verificado')

print('Cargando AV...')
if MODO_8BIT:
    av_model = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AV,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map='auto', trust_remote_code=True,
    )
else:
    av_model = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AV, torch_dtype=torch.bfloat16,
        device_map='cuda:0', trust_remote_code=True,
    )
av_model.eval()
print(f'✓ AV cargado. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')

DTYPE_EMB = torch.bfloat16 if not MODO_8BIT else torch.float16

def cargar_embedding_rapido(checkpoint_dir, dtype):
    index_path = f'{checkpoint_dir}/model.safetensors.index.json'
    if os.path.exists(index_path):
        weight_map = json.load(open(index_path))['weight_map']
        key   = [k for k in weight_map if k.endswith('embed_tokens.weight')][0]
        shard = f'{checkpoint_dir}/{weight_map[key]}'
    else:
        shard = f'{checkpoint_dir}/model.safetensors'
        with safe_open(shard, framework='pt') as f_:
            key = [k for k in f_.keys() if k.endswith('embed_tokens.weight')][0]
    with safe_open(shard, framework='pt') as f_:
        weight = f_.get_tensor(key).to(dtype)
    emb = torch.nn.Embedding(*weight.shape, _weight=weight)
    emb.requires_grad_(False)
    return emb.eval()

embed_layer = cargar_embedding_rapido(CHECKPOINT_AV, DTYPE_EMB)
EMBED_SCALE = 1.0
print('✓ Capa de embedding cargada')

In [ ]:
def verbalizar(v_raw, temperatura=1.0, max_nuevos_tokens=200):
    contenido = PROMPT_TEMPLATE.format(injection_char=INJECTION_CHAR)
    formatted_string = tok_av.apply_chat_template(
        [{'role': 'user', 'content': contenido}],
        tokenize=False,
        add_generation_prompt=True,
    )
    input_ids = tok_av.encode(formatted_string, add_special_tokens=False)

    ids_t  = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        embeds = (embed_layer(ids_t) * EMBED_SCALE).float()

    v      = torch.as_tensor(v_raw, dtype=torch.float32)
    norma  = v.norm().clamp_min(1e-12)
    v_scaled = v * (INJECTION_SCALE / norma)

    inyectado = False
    for p in range(1, len(input_ids) - 1):
        if (input_ids[p]   == INJECTION_TOK_ID and
            input_ids[p-1] == LEFT_NEIGHBOR and
            input_ids[p+1] == RIGHT_NEIGHBOR):
            embeds[0, p] = v_scaled
            inyectado = True
            break
    assert inyectado, 'No se encontró posición de inyección — revisar prompt template'

    device     = next(av_model.parameters()).device
    embeds_dev = embeds.to(device)
    if MODO_8BIT:
        embeds_dev = embeds_dev.bfloat16()

    with torch.no_grad():
        tokens_gen = av_model.generate(
            inputs_embeds=embeds_dev,
            max_new_tokens=max_nuevos_tokens,
            temperature=temperatura,
            do_sample=True,
            pad_token_id=tok_av.eos_token_id,
        )

    texto_gen = tok_av.decode(tokens_gen[0], skip_special_tokens=False)
    m = re.search(r'<explanation>\s*(.*?)\s*</explanation>', texto_gen, re.DOTALL)
    return m.group(1).strip() if m else texto_gen

print('✓ Función verbalizar() lista')

In [ ]:
# ── Loop batch ──────────────────────────────────────────────────────
# Una sola llamada a verbalizar() por entrada (activación ya es mean-pooled [3584])

COLUMNAS = ['id', 'lang', 'grupo', 'tema', 'texto',
            'senales_colombianas', 'hipotesis_nla', 'explicacion_nla']

filas   = []
errores = []
total   = len(metadatos)

print(f'Verbalizando {total} activaciones (~10-30 s por entrada)...\n')

for i, entrada in enumerate(metadatos):
    pid  = entrada['id']
    lang = entrada['lang']
    print(f'[{i+1:3d}/{total}] {pid}_{lang} ...', end=' ', flush=True)

    try:
        ruta_npy = f'{DIR_ACT}/{entrada["archivo_npy"]}'
        v_raw    = np.load(ruta_npy)          # [3584]
        explicacion = verbalizar(v_raw)

        filas.append({
            'id'                 : pid,
            'lang'               : lang,
            'grupo'              : entrada['grupo'],
            'tema'               : entrada['tema'],
            'texto'              : entrada['texto'],
            'senales_colombianas': '|'.join(entrada.get('senales_colombianas', [])),
            'hipotesis_nla'      : entrada.get('hipotesis_nla', ''),
            'explicacion_nla'    : explicacion,
        })
        print(f'✓  {explicacion[:70]}...')

    except Exception as e:
        errores.append({'id': pid, 'lang': lang, 'error': str(e)})
        print(f'✗  ERROR: {e}')

# ── Guardar CSV ─────────────────────────────────────────────────────
with open(CSV_SALIDA, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNAS)
    writer.writeheader()
    writer.writerows(filas)

print(f'\n{"="*55}')
print(f'✓ Verbalizaciones guardadas : {len(filas)}')
print(f'✗ Errores                   : {len(errores)}')
if errores:
    for e in errores:
        print(f'   {e}')
print(f'✓ CSV → {CSV_SALIDA}')
print(f'\n🎉 Etapa 2 completada. Continúa con el análisis de sesgos.')